In [1]:
import math
import shutil
from functools import reduce
from os import mkdir
from pathlib import Path
import random
from time import sleep

import pyarrow as pa
import pyarrow.parquet as pq

import numpy as np
import polars as pl
from tqdm import tqdm


In [2]:
DATA_DIR = Path("/mnt/Fast2T/data/ai-training-set/")
DATASET_DIR = Path(DATA_DIR, "dataset/")

Find embedding
We embedd a text using the usage of stem words, so to find a embedding we combine the top words of:
- web data (to have some relation to our data set)
- ai data (to find patterns like —)
- wikipedia data (to find human patterns)

In [5]:
TOP_K = 128

ai = pl.read_parquet(DATA_DIR / "total" / "ai.parquet")
website = pl.read_parquet(DATA_DIR / "total" / "web.parquet")
wiki = pl.read_parquet(DATA_DIR / "total" / "human.parquet")
grok = pl.read_parquet(DATA_DIR / "total" / "grok.parquet")
cc = pl.read_parquet(DATA_DIR / "total" / "cc.parquet")

common = (
    ai.select("term", pl.col("freq").alias("len_ai"))
    .join(wiki.select("term", pl.col("freq").alias("len_wiki")), on="term")
    .join(website.select("term", pl.col("freq").alias("len_website")), on="term")
    .join(grok.select("term", pl.col("freq").alias("len_grok")), on="term")
    .join(cc.select("term", pl.col("freq").alias("len_cc")), on="term")
    .unique(subset="term")
)

embedding = (
    common
    .with_columns(pl.mean_horizontal("len_ai", "len_wiki", "len_website", "len_grok", "len_cc").alias("len_avg"))
    .top_k(k=TOP_K, by="len_avg")
    .sort("term")
    .select("term")
    .rename({"term": "token"})
    .with_row_index("id")
)

embedding.write_parquet(DATASET_DIR / "embedding.parquet")

del ai, cc, common, grok, website, wiki

embedding_size = int(embedding.count()["id"][0])
embedding

id,token
u32,str
0,"""about"""
1,"""access"""
2,"""add"""
3,"""after"""
4,"""align"""
…,…
123,"""world"""
124,"""year"""
125,"""years"""


Create a dataset based on the embedding

In [6]:
TARGET_DIR = Path(DATA_DIR / "embedded-exploded")
tokens = embedding["token"].to_list()

for data_type in list(Path(DATA_DIR / "exploded").iterdir()):
    data_name = data_type.name

    (TARGET_DIR / data_name).mkdir(parents=True, exist_ok=True)

    for file in tqdm(list(Path(DATA_DIR / "exploded" / data_name).iterdir()), desc=data_name):
        (
            pl
            .scan_parquet(file)
            .select("id", "term", "count")
            .filter(pl.col("term").is_in(tokens))
            .with_columns(total=pl.col("count").sum().over("id"))
            .sink_parquet(TARGET_DIR / data_name / file.name, compression="zstd")
        )

grok: 100%|██████████| 9/9 [00:01<00:00,  4.75it/s]


In [7]:
SOURCE_DIR = Path(DATA_DIR / "embedded-exploded")
TARGET_DIR = Path(DATA_DIR / "sampled")

TARGET_IDS = 6_000_000

for d in list(SOURCE_DIR.iterdir()):
    name = d.name
    files = list(d.glob("*.parquet"))

    total_ids = sum(pl.read_parquet(f, columns=["id"]).n_unique() for f in tqdm(files, desc=f"{name} prescan"))
    frac = min(1.0, TARGET_IDS / total_ids)
    print(f"{name}: {total_ids} unique ids, sampling {frac:.2%}")

    writer = None
    written = 0

    for file in tqdm(files, desc=f"{name} scan"):
        df = pl.read_parquet(file)
        sampled_ids = df.get_column("id").unique().sample(fraction=frac, seed=42)
        filtered = df.filter(df.get_column("id").is_in(sampled_ids.to_list()))

        if len(filtered) > 0:
            if writer is None:
                writer = pq.ParquetWriter(
                    TARGET_DIR / f"{name}.parquet",
                    filtered.to_arrow().schema,
                    compression="zstd",
                )
            writer.write_table(filtered.to_arrow())
            written += len(filtered)
        del df, filtered

    if writer is not None:
        writer.close()
    print(f"{name}: wrote {written} rows from ~{int(total_ids * frac)} ids")

human prescan: 100%|██████████| 180/180 [00:02<00:00, 81.26it/s]


human: 5033457 unique ids, sampling 100.00%


human scan: 100%|██████████| 180/180 [00:14<00:00, 12.35it/s]


human: wrote 214069836 rows from ~5033457 ids


web prescan: 100%|██████████| 1498/1498 [00:17<00:00, 86.61it/s] 


...

cc: wrote 28541402 rows from ~716804 ids


grok prescan: 100%|██████████| 9/9 [00:00<00:00, 12.25it/s]


grok: 811965 unique ids, sampling 100.00%


grok scan: 100%|██████████| 9/9 [00:04<00:00,  2.10it/s]

grok: wrote 64699512 rows from ~811965 ids


In [8]:
SAMPLED_DIR = DATA_DIR / "sampled"
SHARD_DIR = DATA_DIR / "shards"
SHARD_DIR.mkdir(exist_ok=True)

SHARDS = 4

labels = {
    "ai": 1,
    "grok": 1,
    "cc": 0,
    "human": 0
}

for label in labels:
    label_file = SAMPLED_DIR / f"{label}.parquet"

    ids = pl.scan_parquet(label_file).select(pl.col("id").unique()).collect().to_series().shuffle(seed=42)
    chunks = ids.reshape((-1,)).to_list()
    chunk_size = math.ceil(len(chunks) / SHARDS)

    for i in tqdm(range(SHARDS), desc=f"{label} sharding"):
        batch_ids = chunks[i * chunk_size: (i + 1) * chunk_size]
        (
            pl
            .scan_parquet(label_file)
            .filter(pl.col("id").is_in(batch_ids))
            .with_columns(label=labels[label])
            .sink_parquet(SAMPLED_DIR / f"{label}-{i}.parquet", compression="zstd")
        )

human sharding: 100%|██████████| 4/4 [00:19<00:00,  4.80s/it]


In [9]:
SAMPLED_DIR = DATA_DIR / "sampled"
SHARD_DIR = DATA_DIR / "shards"
SHARD_DIR.mkdir(exist_ok=True)

SHARDS = 4
EMBED_DIM = embedding_size
embed_lookup = embedding.rename({"id": "embedding_id", "token": "term"})
source_files = ["ai", "grok", "cc", "human"]

for i in range(SHARDS):
    all_ids = pl.concat([
        pl.read_parquet(SAMPLED_DIR / f"{s}-{i}.parquet", columns=["id"])
        for s in source_files
    ])["id"].unique().sort()

    n = len(all_ids)
    id_lookup = pl.DataFrame({"id": all_ids}).with_row_index("row_idx")

    emb = np.zeros((n, EMBED_DIM), dtype=np.float32)
    lbl = np.zeros(n, dtype=np.int8)

    for s in tqdm(source_files, desc=f"shard {i}"):
        df = pl.read_parquet(SAMPLED_DIR / f"{s}-{i}.parquet")
        df2 = df.join(id_lookup, on="id").join(embed_lookup, on="term")

        emb[
            df2["row_idx"].to_numpy(),
            df2["embedding_id"].to_numpy(),
        ] = (df2["count"].cast(pl.Float32) / df2["total"].cast(pl.Float32)).to_numpy()

        rows = df2.group_by("row_idx").agg(pl.col("label").first()).sort("row_idx")
        lbl[rows["row_idx"].to_numpy()] = rows["label"].to_numpy().astype(np.int8)

        del df, df2, rows

    perm = np.random.default_rng().permutation(n)
    emb, lbl = emb[perm], lbl[perm]

    np.save(SHARD_DIR / f"train_{i}.emb.npy", emb)
    np.save(SHARD_DIR / f"train_{i}.lbl.npy", lbl)

    print(f"train_{i}: {n:,} ids, {emb.nbytes / 1e9:.1f} GB, "
          f"label0={int((lbl == 0).sum()):,} label1={int((lbl == 1).sum()):,}")

    del emb, lbl, id_lookup, all_ids

shard 0: 100%|██████████| 4/4 [00:04<00:00,  1.18s/it]


train_0: 3,079,129 ids, 1.6 GB, label0=1,437,566 label1=1,641,563


shard 1: 100%|██████████| 4/4 [00:03<00:00,  1.07it/s]


train_1: 3,079,129 ids, 1.6 GB, label0=1,437,566 label1=1,641,563


shard 2: 100%|██████████| 4/4 [00:04<00:00,  1.10s/it]


train_2: 3,079,129 ids, 1.6 GB, label0=1,437,566 label1=1,641,563


shard 3: 100%|██████████| 4/4 [00:05<00:00,  1.29s/it]


train_3: 3,079,123 ids, 1.6 GB, label0=1,437,563 label1=1,641,560
